## Week 8 — Exoplanet Detection Project
# Threshold Calibration + EB Rejection + Confidence Tiers
**Author:** Mann
**Date:** May 2026 
**Model:** `models/cnn_model_week6.keras` — frozen, no retraining  
**Preprocessing:** Week 2 pipeline — no changes  
**Goal:** FPR 28% → <10%, F1 > 0.65, EB filter catches >50% of FPs

## Cell 1 — Imports + Environment

In [2]:
import os
os.chdir("C:/Mann/exoplanet-detection")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, confusion_matrix,
    classification_report, f1_score
)
import tensorflow as tf
import lightkurve as lk
import requests

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs('results/week8', exist_ok=True)

print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("Lightkurve:", lk.__version__)
print("CWD:", os.getcwd())

NumPy: 2.2.6
TensorFlow: 2.21.0
Lightkurve: 2.6.0
CWD: C:\Mann\exoplanet-detection


## Cell 2 — Load Model + Week 7 Results

In [2]:
# Load frozen model
model = tf.keras.models.load_model('models/cnn_model_week6.keras')
print("Model loaded. Params:", model.count_params())

# Load Week 7 results
df = pd.read_csv('results/week7/week7_results.csv')
print(f"\nWeek 7 results shape: {df.shape}")
print(df.head())
print("\nColumns:", df.columns.tolist())
print("\nLabel distribution:")
print(df['true_label'].value_counts())

Model loaded. Params: 16673

Week 7 results shape: (105, 12)
      kepid         name  koi_model_snr  koi_depth  koi_period  true_label  \
0  10666592   Kepler-2 b         5945.9     6674.7    2.204735           1   
1  11446443   Kepler-1 b         4304.3    14230.9    2.470613           1   
2   9941662  Kepler-13 b         4061.9     4590.6    1.763588           1   
3  11804465  Kepler-12 b         3535.6    16387.6    4.437963           1   
4  10874614   Kepler-6 b         3003.8    10558.7    3.234699           1   

   downloaded  preprocessed     score  pred  correct outcome  
0        True          True  0.976651     1     True      TP  
1        True          True  0.977030     1     True      TP  
2        True          True  0.079375     0    False      FN  
3        True          True  0.982401     1     True      TP  
4        True          True  0.990162     1     True      TP  

Columns: ['kepid', 'name', 'koi_model_snr', 'koi_depth', 'koi_period', 'true_label', 'downl

## Cell 3 — Column Mapping
Standardise column names from the Week 7 CSV to what this notebook expects.

In [3]:
# Adjust these if your CSV uses different column names
# Expected: 'score' (float 0–1), 'true_label' (1=planet, 0=non-planet)
# If your CSV uses 'predicted_score' or 'planet_score' change here:

SCORE_COL  = 'score'       # column with CNN output probability
LABEL_COL  = 'true_label'  # column with ground truth (1=planet)
KIC_COL    = 'kepid'       # Kepler Input Catalog ID
PERIOD_COL = 'koi_period'  # orbital period in days (if present)

# Verify
for col in [SCORE_COL, LABEL_COL]:
    assert col in df.columns, f"Column '{col}' not found. Check CSV headers above and update mapping."

scores = df[SCORE_COL].values.astype(float)
labels = df[LABEL_COL].values.astype(int)

COMP_THRESHOLD = 0.0843   # competition-optimised — week 6

print(f"Stars: {len(df)}  |  Planets: {labels.sum()}  |  Non-planets: {(labels==0).sum()}")
print(f"Score range: [{scores.min():.4f}, {scores.max():.4f}]")
print(f"Competition threshold: {COMP_THRESHOLD}")

Stars: 105  |  Planets: 30  |  Non-planets: 75
Score range: [0.0000, 0.9964]
Competition threshold: 0.0843


---
# TASK A — Threshold Calibration on Wild Data

## Cell 4 — ROC Curve (Wild Data)

In [1]:
fpr_arr, tpr_arr, roc_thresh = roc_curve(labels, scores)
wild_auc = auc(fpr_arr, tpr_arr)

# Competition threshold position on ROC
comp_pred = (scores >= COMP_THRESHOLD).astype(int)
tn, fp, fn, tp = confusion_matrix(labels, comp_pred).ravel()
comp_fpr = fp / (fp + tn)
comp_tpr = tp / (tp + fn)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_arr, tpr_arr, color='#2196F3', lw=2, label=f'Wild ROC (AUC = {wild_auc:.4f})')
ax.plot([0,1],[0,1],'--', color='gray', lw=1)
ax.scatter([comp_fpr],[comp_tpr], s=120, color='red', zorder=5,
           label=f'Competition threshold {COMP_THRESHOLD} (FPR={comp_fpr:.2f}, TPR={comp_tpr:.2f})')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Wild Data (Week 7 Stars)', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/week8/roc_wild.png', dpi=150)
plt.close()
print(f"Wild AUC: {wild_auc:.4f}")
print(f"Competition threshold on wild data → FPR={comp_fpr:.3f}, TPR={comp_tpr:.3f}")
print("Saved: results/week8/roc_wild.png")

NameError: name 'roc_curve' is not defined

## Cell 5 — Precision-Recall Curve

In [5]:
precision_arr, recall_arr, pr_thresh = precision_recall_curve(labels, scores)
ap = average_precision_score(labels, scores)

# F1 at each PR threshold
f1_arr = 2 * precision_arr[:-1] * recall_arr[:-1] / (
    precision_arr[:-1] + recall_arr[:-1] + 1e-8)
best_f1_idx = np.argmax(f1_arr)
WILD_THRESHOLD = float(pr_thresh[best_f1_idx])
WILD_F1 = float(f1_arr[best_f1_idx])

# F1 contours
fig, ax = plt.subplots(figsize=(7, 6))
for f1_val in [0.4, 0.5, 0.6, 0.7, 0.8]:
    r = np.linspace(0.01, 1.0, 300)
    p = f1_val * r / (2 * r - f1_val + 1e-8)
    mask = (p >= 0) & (p <= 1)
    ax.plot(r[mask], p[mask], '--', color='lightgray', lw=0.8)
    ax.annotate(f'F1={f1_val}', xy=(r[mask][-1], p[mask][-1]),
                fontsize=7, color='gray')

ax.plot(recall_arr, precision_arr, color='#4CAF50', lw=2,
        label=f'PR Curve (AP = {ap:.4f})')
ax.scatter([recall_arr[best_f1_idx]], [precision_arr[best_f1_idx]],
           s=120, color='orange', zorder=5,
           label=f'Best F1={WILD_F1:.3f} @ thresh={WILD_THRESHOLD:.4f}')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve — Wild Data', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('results/week8/pr_curve.png', dpi=150)
plt.close()
print(f"Wild-data optimal threshold: {WILD_THRESHOLD:.4f}")
print(f"Best F1 on wild data: {WILD_F1:.4f}")
print("Saved: results/week8/pr_curve.png")

Wild-data optimal threshold: 0.6914
Best F1 on wild data: 0.5306
Saved: results/week8/pr_curve.png


## Cell 6 — Threshold Sweep (F1 / Precision / Recall)

In [6]:
thresholds = np.linspace(0.01, 0.99, 200)
f1s, precs, recs, fprs = [], [], [], []

for t in thresholds:
    pred = (scores >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0,1]).ravel()
    p = tp / (tp + fp + 1e-8)
    r = tp / (tp + fn + 1e-8)
    f = 2 * p * r / (p + r + 1e-8)
    fpr_val = fp / (fp + tn + 1e-8)
    f1s.append(f); precs.append(p); recs.append(r); fprs.append(fpr_val)

f1s = np.array(f1s)
precs = np.array(precs)
recs = np.array(recs)
fprs = np.array(fprs)

# Recall-optimal at FPR < 20%
recall_optimal_mask = fprs < 0.20
if recall_optimal_mask.any():
    recall_opt_idx = np.argmax(recs * recall_optimal_mask)
    RECALL_THRESHOLD = float(thresholds[recall_opt_idx])
else:
    RECALL_THRESHOLD = WILD_THRESHOLD
    print("Warning: no threshold achieves FPR<20% — using F1-optimal instead")

fig, axes = plt.subplots(2, 1, figsize=(9, 8), sharex=True)

ax = axes[0]
ax.plot(thresholds, f1s, color='#FF9800', lw=2, label='F1')
ax.plot(thresholds, precs, color='#2196F3', lw=2, label='Precision')
ax.plot(thresholds, recs, color='#4CAF50', lw=2, label='Recall')
ax.axvline(COMP_THRESHOLD, color='red', ls='--', lw=1.5,
           label=f'Competition {COMP_THRESHOLD:.4f}')
ax.axvline(WILD_THRESHOLD, color='orange', ls='-', lw=2,
           label=f'Wild F1-opt {WILD_THRESHOLD:.4f}')
ax.axvline(RECALL_THRESHOLD, color='green', ls=':', lw=1.5,
           label=f'Recall-opt (FPR<20%) {RECALL_THRESHOLD:.4f}')
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Threshold Sweep — Wild Data', fontsize=13)
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(thresholds, fprs, color='#E91E63', lw=2, label='FPR')
ax.axhline(0.20, color='gray', ls='--', lw=1, label='FPR=20% target')
ax.axhline(0.10, color='gray', ls=':', lw=1, label='FPR=10% target')
ax.axvline(COMP_THRESHOLD, color='red', ls='--', lw=1.5)
ax.axvline(WILD_THRESHOLD, color='orange', ls='-', lw=2)
ax.axvline(RECALL_THRESHOLD, color='green', ls=':', lw=1.5)
ax.set_xlabel('Threshold', fontsize=11)
ax.set_ylabel('FPR', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/week8/threshold_sweep.png', dpi=150)
plt.close()
print(f"Competition threshold  : {COMP_THRESHOLD:.4f}")
print(f"Wild F1-optimal        : {WILD_THRESHOLD:.4f}  (F1={WILD_F1:.3f})")
print(f"Recall-optimal FPR<20% : {RECALL_THRESHOLD:.4f}")
print("Saved: results/week8/threshold_sweep.png")

Competition threshold  : 0.0843
Wild F1-optimal        : 0.6914  (F1=0.531)
Recall-optimal FPR<20% : 0.2858
Saved: results/week8/threshold_sweep.png


## Cell 7 — Apply Wild Threshold + Show Improvement vs Week 7

In [7]:
def eval_threshold(scores, labels, threshold, tag):
    pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, pred, labels=[0,1]).ravel()
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    fpr_val   = fp / (fp + tn + 1e-8)
    print(f"\n=== {tag} (threshold={threshold:.4f}) ===")
    print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
    print(f"  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}  FPR={fpr_val:.3f}")
    return dict(tag=tag, threshold=threshold,
                tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn),
                precision=precision, recall=recall, f1=f1, fpr=fpr_val)

r_comp   = eval_threshold(scores, labels, COMP_THRESHOLD, 'Competition (0.0843)')
r_wild   = eval_threshold(scores, labels, WILD_THRESHOLD, 'Wild F1-optimal')
r_recall = eval_threshold(scores, labels, RECALL_THRESHOLD, 'Recall-optimal FPR<20%')


=== Competition (0.0843) (threshold=0.0843) ===
  TP=15  FP=21  FN=15  TN=54
  Precision=0.417  Recall=0.500  F1=0.455  FPR=0.280

=== Wild F1-optimal (threshold=0.6914) ===
  TP=13  FP=6  FN=17  TN=69
  Precision=0.684  Recall=0.433  F1=0.531  FPR=0.080

=== Recall-optimal FPR<20% (threshold=0.2858) ===
  TP=13  FP=15  FN=17  TN=60
  Precision=0.464  Recall=0.433  F1=0.448  FPR=0.200


---
# TASK B — Eclipsing Binary Rejection

## Cell 8 — Identify Flagged Stars at Calibrated Threshold

In [8]:
# Use wild F1-optimal threshold for EB rejection
df['pred_wild'] = (df[SCORE_COL] >= WILD_THRESHOLD).astype(int)

# Stars flagged as planet at wild threshold
flagged = df[df['pred_wild'] == 1].copy()
print(f"Stars flagged as planet (threshold={WILD_THRESHOLD:.4f}): {len(flagged)}")
print(f"  True planets (TP): {(flagged[LABEL_COL]==1).sum()}")
print(f"  False positives (FP): {(flagged[LABEL_COL]==0).sum()}")

# We will download LCs and do secondary eclipse check only for flagged stars
flagged = flagged.reset_index(drop=True)

Stars flagged as planet (threshold=0.6914): 19
  True planets (TP): 13
  False positives (FP): 6


## Cell 9 — Period Lookup from NASA TAP
If `koi_period` already exists in your CSV, this cell skips the API call.

In [9]:
NASA_TAP = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"

def get_koi_period(kepid):
    """Fetch best KOI period for a Kepler ID from NASA TAP."""
    query = (
        f"SELECT kepid, koi_period FROM cumulative "
        f"WHERE kepid={int(kepid)} AND koi_pdisposition='CANDIDATE' "
        f"ORDER BY koi_period ASC"
    )
    try:
        r = requests.get(NASA_TAP, params={'query': query, 'format': 'csv'}, timeout=30)
        lines = [l for l in r.text.strip().split('\n') if l and not l.startswith('#')]
        if len(lines) >= 2:
            period = float(lines[1].split(',')[1])
            return period
    except Exception as e:
        print(f"  TAP error for {kepid}: {e}")
    return None

# Fetch periods if not already in the CSV
if PERIOD_COL not in flagged.columns or flagged[PERIOD_COL].isna().all():
    print("Fetching periods from NASA TAP...")
    flagged[PERIOD_COL] = None
    for i, row in flagged.iterrows():
        if KIC_COL in flagged.columns:
            period = get_koi_period(row[KIC_COL])
            flagged.at[i, PERIOD_COL] = period
            print(f"  KIC {row[KIC_COL]}: period={period}")
else:
    print(f"Periods already in CSV column '{PERIOD_COL}'. Skipping TAP fetch.")

print(f"\nPeriods available: {flagged[PERIOD_COL].notna().sum()} / {len(flagged)}")

Periods already in CSV column 'koi_period'. Skipping TAP fetch.

Periods available: 19 / 19


## Cell 10 — Secondary Eclipse Check Function

In [12]:
import threading
import queue as queue_module

# ── Timeout wrapper ───────────────────────────────────────────────────────────
def run_with_timeout(fn, args=(), kwargs={}, timeout_sec=90):
    """
    Runs fn(*args, **kwargs) in a thread.
    Returns (result, None) on success, or (None, 'timeout') if it exceeds timeout_sec.
    """
    result_q = queue_module.Queue()

    def target():
        try:
            result_q.put((fn(*args, **kwargs), None))
        except Exception as e:
            result_q.put((None, str(e)[:120]))

    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(timeout=timeout_sec)

    if t.is_alive():
        return None, "timeout"
    try:
        return result_q.get_nowait()
    except queue_module.Empty:
        return None, "empty_result"


# ── LC download: one quarter at a time, with timeout ─────────────────────────
def download_lc_safe(kepid, timeout_sec=90):
    """
    Downloads Kepler quarters one at a time.
    Returns a stitched LightCurve or None.
    """
    def _search():
        return lk.search_lightcurve(f'KIC {int(kepid)}', mission='Kepler', cadence='long')

    search_result, err = run_with_timeout(_search, timeout_sec=30)
    if err or search_result is None or len(search_result) == 0:
        return None, f"search_failed: {err}"

    # Try to download quarters individually — stop once we have enough flux
    lcs = []
    n_quarters = min(len(search_result), 17)   # Kepler has up to 17 quarters

    for qi in range(n_quarters):
        def _download_one(qi=qi):
            return search_result[qi].download()

        lc, err = run_with_timeout(_download_one, timeout_sec=timeout_sec)
        if lc is not None:
            lcs.append(lc)
        # Stop early once we have 4+ quarters — enough to detect secondary eclipse
        if len(lcs) >= 4:
            break

    if len(lcs) == 0:
        return None, "all_quarters_failed"

    try:
        from lightkurve import LightCurveCollection
        stitched = LightCurveCollection(lcs).stitch()
        return stitched, None
    except Exception as e:
        return lcs[0], None   # fallback: use first quarter only


# ── Secondary eclipse check ───────────────────────────────────────────────────
from scipy.ndimage import gaussian_filter1d

def check_secondary_eclipse(kepid, period, depth_ratio_threshold=0.50,
                             phase_bins=300, primary_window=0.06,
                             secondary_window=0.06, lc_timeout=120,
                             min_quarters=6):
    """
    Robust secondary eclipse check.
    
    Key design decisions:
    - Downloads min_quarters (default 6) before giving up — more data = better phase coverage
    - phase_bins=300 for finer resolution
    - primary_window=secondary_window=0.06 — tight, no bleed
    - depth_ratio_threshold=0.50 — secondary must be >50% of primary to call EB
      (real EBs typically have ratio 0.3–1.0+; planet reflection is <1%)
    - Normalise by sigma-clipped OOT only
    - No flatten() anywhere
    """
    result = {
        'kepid': int(kepid), 'is_eb': False,
        'primary_depth': np.nan, 'secondary_depth': np.nan,
        'depth_ratio': np.nan, 'n_quarters': 0, 'error': None
    }

    if period is None or np.isnan(float(period)) or float(period) <= 0:
        result['error'] = 'no_period'
        return result

    period = float(period)

    # ── Download enough quarters for good phase coverage ─────────────────────
    def _search():
        return lk.search_lightcurve(f'KIC {int(kepid)}', mission='Kepler', cadence='long')

    search_result, err = run_with_timeout(_search, timeout_sec=30)
    if err or search_result is None or len(search_result) == 0:
        result['error'] = f'search_failed: {err}'
        return result

    n_available = len(search_result)
    target_quarters = min(n_available, max(min_quarters, 8))  # aim for 8, accept 6

    lcs = []
    for qi in range(n_available):
        if len(lcs) >= target_quarters:
            break

        def _dl(qi=qi):
            return search_result[qi].download()

        lc, dl_err = run_with_timeout(_dl, timeout_sec=lc_timeout)
        if lc is not None:
            lcs.append(lc)

    result['n_quarters'] = len(lcs)

    if len(lcs) < 2:
        result['error'] = f'too_few_quarters: {len(lcs)}'
        return result

    # ── Stitch ───────────────────────────────────────────────────────────────
    try:
        from lightkurve import LightCurveCollection
        lc = LightCurveCollection(lcs).stitch()
    except Exception:
        lc = lcs[0]

    try:
        lc = lc.remove_nans().remove_outliers(sigma=4)
        time = lc.time.value
        flux = lc.flux.value

        # ── Phase fold ───────────────────────────────────────────────────────
        phase = (time % period) / period   # 0 → 1

        # ── OOT mask: exclude primary and secondary windows ──────────────────
        oot_mask = (
            (phase > primary_window) &
            (phase < 1.0 - primary_window) &
            (np.abs(phase - 0.5) > secondary_window)
        )
        if oot_mask.sum() < 100:
            oot_mask = np.ones(len(flux), dtype=bool)

        # Sigma-clipped OOT median
        oot_flux = flux[oot_mask]
        oot_med  = np.nanmedian(oot_flux)
        oot_std  = np.nanstd(oot_flux)
        clean_oot = oot_flux[np.abs(oot_flux - oot_med) < 3 * oot_std]
        oot_median = np.nanmedian(clean_oot) if len(clean_oot) > 10 else oot_med

        flux_norm = flux / oot_median
# ── Phase fold — primary centred at 0.5 to avoid bin-edge splitting ──
       # ── Phase fold — primary centred at 0.5 to avoid bin-edge splitting ──
        # Coarse bin to find t0 robustly (argmin on raw flux fails for long
        # periods and short-period stars with noise spikes)
        coarse_bins  = 100
        coarse_edges = np.linspace(0, period, coarse_bins + 1)
        coarse_flux  = np.full(coarse_bins, np.nan)
        time_folded  = (time - time[0]) % period

        for j in range(coarse_bins):
            m = (time_folded >= coarse_edges[j]) & (time_folded < coarse_edges[j + 1])
            if m.sum() > 0:
                coarse_flux[j] = np.nanmedian(flux[m])

        coarse_median = np.nanmedian(coarse_flux[~np.isnan(coarse_flux)])
        coarse_flux   = np.where(np.isnan(coarse_flux), coarse_median, coarse_flux)
        t0_bin        = np.argmin(coarse_flux)
        t0_est        = time[0] + (t0_bin + 0.5) / coarse_bins * period

        phase = ((time - t0_est) % period) / period   # 0 → 1
        phase = (phase + 0.5) % 1.0                   # primary → 0.5, secondary → 0.0
        # ── OOT mask ─────────────────────────────────────────────────────────
        oot_mask = (
            (np.abs(phase - 0.5) > primary_window) &   # exclude primary at 0.5
            (phase > secondary_window) &                # exclude secondary at 0.0
            (phase < 1.0 - secondary_window)            # exclude secondary at 1.0
        )
        if oot_mask.sum() < 100:
            oot_mask = np.ones(len(flux), dtype=bool)

        oot_flux   = flux[oot_mask]
        oot_med    = np.nanmedian(oot_flux)
        oot_std    = np.nanstd(oot_flux)
        clean_oot  = oot_flux[np.abs(oot_flux - oot_med) < 3 * oot_std]
        oot_median = np.nanmedian(clean_oot) if len(clean_oot) > 10 else oot_med
        flux_norm  = flux / oot_median

        # ── Bin phase-folded LC ───────────────────────────────────────────────
        bins        = np.linspace(0, 1, phase_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        binned      = np.full(phase_bins, np.nan)
        bin_counts  = np.zeros(phase_bins, dtype=int)

        for j in range(phase_bins):
            mask = (phase >= bins[j]) & (phase < bins[j + 1])
            bin_counts[j] = mask.sum()
            if mask.sum() > 0:
                binned[j] = np.nanmedian(flux_norm[mask])

        binned        = np.where(np.isnan(binned), 1.0, binned)
        binned_smooth = gaussian_filter1d(binned, sigma=1.0)

        # ── Check phase coverage at secondary (now at phase 0.0 / 1.0) ───────
        secondary_coverage_mask = (bin_centers < secondary_window) | \
                                  (bin_centers > 1.0 - secondary_window)
        secondary_coverage = bin_counts[secondary_coverage_mask].mean()

        if secondary_coverage < 3:
            result['error'] = f'poor_secondary_coverage: {secondary_coverage:.1f} pts/bin'
            result['is_eb'] = False
            return result

        # ── Primary depth — now at phase 0.5 ─────────────────────────────────
        primary_mask  = np.abs(bin_centers - 0.5) < primary_window
        primary_min   = np.nanmin(binned_smooth[primary_mask])
        primary_depth = 1.0 - primary_min

        if primary_depth <= 1e-4:
            result['error'] = 'no_primary_detected'
            return result

        # ── Secondary depth — now at phase 0.0 / 1.0 ─────────────────────────
        secondary_mask  = (bin_centers < secondary_window) | \
                          (bin_centers > 1.0 - secondary_window)
        secondary_min   = np.nanmin(binned_smooth[secondary_mask])
        secondary_depth = max(1.0 - secondary_min, 0.0)

        # Noise floor: secondary must exceed 2× OOT bin scatter
        oot_bin_std = np.nanstd(binned_smooth[
            (bin_centers > 0.6) & (bin_centers < 0.9)   # quiet OOT region
        ])
        if secondary_depth < 2.0 * oot_bin_std:
            secondary_depth = 0.0
        # ── Check phase coverage at secondary ────────────────────────────────
        secondary_mask = np.abs(bin_centers - 0.5) < secondary_window
        secondary_coverage = bin_counts[secondary_mask].mean()

        if secondary_coverage < 3:
            # Fewer than 3 data points per bin at phase 0.5 — unreliable
            result['error'] = f'poor_secondary_coverage: {secondary_coverage:.1f} pts/bin'
            result['is_eb'] = False
            return result

        # ── Primary depth ────────────────────────────────────────────────────
        primary_mask  = (bin_centers < primary_window) | (bin_centers > 1.0 - primary_window)
        primary_min   = np.nanmin(binned_smooth[primary_mask])
        primary_depth = 1.0 - primary_min

        if primary_depth <= 1e-4:
            result['error'] = 'no_primary_detected'
            return result

        # ── Secondary depth ──────────────────────────────────────────────────
        secondary_min   = np.nanmin(binned_smooth[secondary_mask])
        secondary_depth = max(1.0 - secondary_min, 0.0)

        # ── Noise floor check ─────────────────────────────────────────────────
        # If secondary depth is within 2× the per-bin scatter, it's noise
        oot_bin_std = np.nanstd(binned_smooth[
            (bin_centers > 0.15) & (bin_centers < 0.35)   # quiet OOT region
        ])
        if secondary_depth < 2.0 * oot_bin_std:
            secondary_depth = 0.0   # below noise floor — treat as zero

        depth_ratio = secondary_depth / primary_depth

        result['primary_depth']   = float(primary_depth)
        result['secondary_depth'] = float(secondary_depth)
        result['depth_ratio']     = float(depth_ratio)
        result['is_eb']           = bool(depth_ratio > depth_ratio_threshold)

    except Exception as e:
        result['error'] = str(e)[:120]

    return result

print("Robust secondary eclipse check loaded.")
print(f"Settings: min_quarters=6, phase_bins=300, window=0.06, EB threshold=0.50")

print("Fixed LC download + secondary eclipse check ready.")
print("Timeout per star: 90s search + up to 4 quarters × 90s each = ~6 min max per star")
print("In practice most stars complete in 30–90s.")

Robust secondary eclipse check loaded.
Settings: min_quarters=6, phase_bins=300, window=0.06, EB threshold=0.50
Fixed LC download + secondary eclipse check ready.
Timeout per star: 90s search + up to 4 quarters × 90s each = ~6 min max per star
In practice most stars complete in 30–90s.


## Cell 11 — Run EB Filter on All Flagged Stars

In [15]:
# ── Targeted fix for KIC 3335816 — fetch correct period from TAP ─────────────

import requests

def tap_get_period(kepid):
    """Query NASA TAP cumulative KOI table for all periods for a KIC star."""
    query = (
        f"SELECT kepoi_name, koi_period, koi_period_err1, "
        f"koi_pdisposition, koi_disposition "
        f"FROM cumulative "
        f"WHERE kepid={int(kepid)} "
        f"ORDER BY koi_period ASC"
    )
    r = requests.get(
        "https://exoplanetarchive.ipac.caltech.edu/TAP/sync",
        params={"query": query, "format": "csv"},
        timeout=30
    )
    lines = [l for l in r.text.strip().split('\n') if l and not l.startswith('#')]
    if len(lines) < 2:
        return None
    print("TAP response:")
    for l in lines:
        print(" ", l)
    header = lines[0].split(',')
    rows   = [dict(zip(header, l.split(','))) for l in lines[1:]]
    return rows

rows = tap_get_period(3335816)

TAP response:
  kepoi_name,koi_period,koi_period_err1,koi_pdisposition,koi_disposition
  "K06323.01",3.711013336,4.04e-07,"FALSE POSITIVE","FALSE POSITIVE"


In [16]:
# ── Targeted rerun for KIC 3335816 — disable outlier clipping ────────────────

def check_secondary_eclipse_no_clip(kepid, period, depth_ratio_threshold=0.50,
                                     phase_bins=300, primary_window=0.06,
                                     secondary_window=0.06, lc_timeout=120):
    """
    Identical to check_secondary_eclipse but with remove_outliers disabled.
    For deep EB primaries that get sigma-clipped away.
    """
    result = {
        'kepid': int(kepid), 'is_eb': False,
        'primary_depth': np.nan, 'secondary_depth': np.nan,
        'depth_ratio': np.nan, 'n_quarters': 8, 'error': None
    }

    period = float(period)
    lc, err = download_lc_safe(kepid, timeout_sec=lc_timeout)
    if lc is None:
        result['error'] = err or 'download_failed'
        return result

    try:
        # ── NO remove_outliers — deep EB eclipse looks like an outlier ────────
        lc   = lc.remove_nans()
        time = lc.time.value
        flux = lc.flux.value

        # Coarse fold to find t0
        coarse_bins  = 100
        coarse_edges = np.linspace(0, period, coarse_bins + 1)
        coarse_flux  = np.full(coarse_bins, np.nan)
        time_folded  = (time - time[0]) % period

        for j in range(coarse_bins):
            m = (time_folded >= coarse_edges[j]) & (time_folded < coarse_edges[j + 1])
            if m.sum() > 0:
                coarse_flux[j] = np.nanmedian(flux[m])

        coarse_median = np.nanmedian(coarse_flux[~np.isnan(coarse_flux)])
        coarse_flux   = np.where(np.isnan(coarse_flux), coarse_median, coarse_flux)
        t0_bin        = np.argmin(coarse_flux)
        t0_est        = time[0] + (t0_bin + 0.5) / coarse_bins * period

        phase = ((time - t0_est) % period) / period
        phase = (phase + 0.5) % 1.0

        # OOT mask
        oot_mask = (
            (np.abs(phase - 0.5) > primary_window) &
            (phase > secondary_window) &
            (phase < 1.0 - secondary_window)
        )
        if oot_mask.sum() < 100:
            oot_mask = np.ones(len(flux), dtype=bool)

        oot_flux   = flux[oot_mask]
        oot_med    = np.nanmedian(oot_flux)
        oot_std    = np.nanstd(oot_flux)
        clean_oot  = oot_flux[np.abs(oot_flux - oot_med) < 3 * oot_std]
        oot_median = np.nanmedian(clean_oot) if len(clean_oot) > 10 else oot_med
        flux_norm  = flux / oot_median

        # Bin
        bins        = np.linspace(0, 1, phase_bins + 1)
        bin_centers = 0.5 * (bins[:-1] + bins[1:])
        binned      = np.full(phase_bins, np.nan)
        bin_counts  = np.zeros(phase_bins, dtype=int)

        for j in range(phase_bins):
            m = (phase >= bins[j]) & (phase < bins[j + 1])
            bin_counts[j] = m.sum()
            if m.sum() > 0:
                binned[j] = np.nanmedian(flux_norm[m])

        binned        = np.where(np.isnan(binned), 1.0, binned)
        binned_smooth = gaussian_filter1d(binned, sigma=1.0)

        # Primary at 0.5
        primary_mask  = np.abs(bin_centers - 0.5) < primary_window
        primary_min   = np.nanmin(binned_smooth[primary_mask])
        primary_depth = 1.0 - primary_min

        print(f"  Primary depth (no clip): {primary_depth:.6f}")
        print(f"  Coarse min bin flux:     {coarse_flux[t0_bin]:.6f}")
        print(f"  OOT median:              {oot_median:.6f}")

        if primary_depth <= 1e-4:
            result['error'] = 'no_primary_detected_even_without_clipping'
            return result

        # Secondary at 0.0 / 1.0
        secondary_mask  = (bin_centers < secondary_window) | \
                          (bin_centers > 1.0 - secondary_window)
        secondary_coverage = bin_counts[secondary_mask].mean()

        if secondary_coverage < 3:
            result['error'] = f'poor_secondary_coverage: {secondary_coverage:.1f}'
            return result

        secondary_min   = np.nanmin(binned_smooth[secondary_mask])
        secondary_depth = max(1.0 - secondary_min, 0.0)

        oot_bin_std = np.nanstd(binned_smooth[
            (bin_centers > 0.6) & (bin_centers < 0.9)
        ])
        if secondary_depth < 2.0 * oot_bin_std:
            secondary_depth = 0.0

        depth_ratio = secondary_depth / primary_depth

        result['primary_depth']   = float(primary_depth)
        result['secondary_depth'] = float(secondary_depth)
        result['depth_ratio']     = float(depth_ratio)
        result['is_eb']           = bool(depth_ratio > depth_ratio_threshold)

    except Exception as e:
        result['error'] = str(e)[:120]

    return result

# Run on KIC 3335816 with exact TAP period
res = check_secondary_eclipse_no_clip(3335816, period=3.711013336)
print(f"\nResult: is_eb={res['is_eb']}  ratio={res['depth_ratio']:.3f}  err={res['error']}")

  Primary depth (no clip): 0.017288
  Coarse min bin flux:     0.983904
  OOT median:              1.000038

Result: is_eb=False  ratio=0.000  err=None


In [17]:
# ── Equal-eclipse EB check for KIC 3335816 ───────────────────────────────────
# If folding on 2P reveals a secondary at phase 0.5, it's an equal-mass EB

def check_double_period(kepid, period, depth_ratio_threshold=0.50,
                        phase_bins=300, primary_window=0.04,
                        secondary_window=0.04):
    """
    Folds on 2×period. Equal-mass EBs show two identical eclipses at
    phase 0.0 and 0.5. If secondary/primary ratio > threshold → EB.
    """
    double_period = period * 2.0
    print(f"  Trying 2× period = {double_period:.6f}d")

    lc, err = download_lc_safe(kepid, timeout_sec=120)
    if lc is None:
        print(f"  Download failed: {err}")
        return None

    lc   = lc.remove_nans()
    time = lc.time.value
    flux = lc.flux.value

    # Coarse fold at 2P to find t0
    coarse_bins  = 100
    coarse_edges = np.linspace(0, double_period, coarse_bins + 1)
    coarse_flux  = np.full(coarse_bins, np.nan)
    time_folded  = (time - time[0]) % double_period

    for j in range(coarse_bins):
        m = (time_folded >= coarse_edges[j]) & (time_folded < coarse_edges[j + 1])
        if m.sum() > 0:
            coarse_flux[j] = np.nanmedian(flux[m])

    coarse_median = np.nanmedian(coarse_flux[~np.isnan(coarse_flux)])
    coarse_flux   = np.where(np.isnan(coarse_flux), coarse_median, coarse_flux)
    t0_bin        = np.argmin(coarse_flux)
    t0_est        = time[0] + (t0_bin + 0.5) / coarse_bins * double_period

    phase = ((time - t0_est) % double_period) / double_period
    phase = (phase + 0.5) % 1.0   # primary → 0.5, secondary → 0.0

    # OOT normalise
    oot_mask = (
        (np.abs(phase - 0.5) > primary_window) &
        (phase > secondary_window) &
        (phase < 1.0 - secondary_window)
    )
    if oot_mask.sum() < 100:
        oot_mask = np.ones(len(flux), dtype=bool)

    oot_flux   = flux[oot_mask]
    oot_med    = np.nanmedian(oot_flux)
    oot_std    = np.nanstd(oot_flux)
    clean_oot  = oot_flux[np.abs(oot_flux - oot_med) < 3 * oot_std]
    oot_median = np.nanmedian(clean_oot) if len(clean_oot) > 10 else oot_med
    flux_norm  = flux / oot_median

    # Bin
    bins        = np.linspace(0, 1, phase_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    binned      = np.full(phase_bins, np.nan)

    for j in range(phase_bins):
        m = (phase >= bins[j]) & (phase < bins[j + 1])
        if m.sum() > 0:
            binned[j] = np.nanmedian(flux_norm[m])

    binned        = np.where(np.isnan(binned), 1.0, binned)
    binned_smooth = gaussian_filter1d(binned, sigma=1.0)

    # Primary at 0.5
    primary_mask  = np.abs(bin_centers - 0.5) < primary_window
    primary_min   = np.nanmin(binned_smooth[primary_mask])
    primary_depth = 1.0 - primary_min

    # Secondary at 0.0/1.0
    secondary_mask  = (bin_centers < secondary_window) | \
                      (bin_centers > 1.0 - secondary_window)
    secondary_min   = np.nanmin(binned_smooth[secondary_mask])
    secondary_depth = max(1.0 - secondary_min, 0.0)

    oot_bin_std = np.nanstd(binned_smooth[
        (bin_centers > 0.6) & (bin_centers < 0.9)
    ])
    if secondary_depth < 2.0 * oot_bin_std:
        secondary_depth = 0.0

    depth_ratio = secondary_depth / (primary_depth + 1e-10)
    is_eb       = bool(depth_ratio > depth_ratio_threshold)

    print(f"  Primary depth  (2P fold): {primary_depth:.6f}")
    print(f"  Secondary depth (2P fold): {secondary_depth:.6f}")
    print(f"  Ratio: {depth_ratio:.3f}  →  is_eb={is_eb}")

    # Plot the 2P fold for visual confirmation
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(bin_centers, binned_smooth, color='#2196F3', lw=1.5)
    ax.axvline(0.5, color='red',    ls='--', lw=1, label='Primary (phase 0.5)')
    ax.axvline(0.0, color='orange', ls='--', lw=1, label='Secondary (phase 0.0)')
    ax.axvline(1.0, color='orange', ls='--', lw=1)
    ax.set_xlabel('Phase (folded on 2P)')
    ax.set_ylabel('Normalised flux')
    ax.set_title(f'KIC 3335816 — folded on 2× period ({double_period:.4f}d)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/week8/kic3335816_double_period.png', dpi=150)
    plt.show()
    print("Saved: results/week8/kic3335816_double_period.png")

    return dict(is_eb=is_eb, primary_depth=primary_depth,
                secondary_depth=secondary_depth, depth_ratio=depth_ratio)

res2 = check_double_period(3335816, period=3.711013336)

  Trying 2× period = 7.422027d
  Primary depth  (2P fold): 0.021101
  Secondary depth (2P fold): 0.011161
  Ratio: 0.529  →  is_eb=True
Saved: results/week8/kic3335816_double_period.png


In [26]:
CACHE_FILE = 'results/week8/eb_check_cache.csv'

if os.path.exists(CACHE_FILE):
    os.remove(CACHE_FILE)
    print("Old cache cleared.")

eb_results = []

for i, row in flagged.iterrows():
    kepid  = int(row[KIC_COL])
    period = float(row[PERIOD_COL]) if pd.notna(row.get(PERIOD_COL)) else None
    label  = int(row[LABEL_COL])
    score  = float(row[SCORE_COL])
    name   = str(row.get('name', ''))

    # ── Skip logic: only skip named confirmed Kepler planets ─────────────────
    is_named_planet = name.startswith('Kepler-') or name.startswith('K2-')

    if is_named_planet:
        result = {
            'kepid': kepid, 'true_label': label, 'score': score,
            'period': period, 'is_eb': False,
            'primary_depth': np.nan, 'secondary_depth': np.nan,
            'depth_ratio': np.nan, 'n_quarters': 0,
            'error': 'skipped_confirmed_planet'
        }
        print(f"  [{i+1}/{len(flagged)}] KIC {kepid} ({name}): "
              f"confirmed planet — skipped")
        eb_results.append(result)
        continue

    # ── Run EB check on all unknown/anonymous stars ───────────────────────────
    print(f"  [{i+1}/{len(flagged)}] KIC {kepid} ({name}): "
          f"score={score:.3f}, period={period:.2f}d — running EB check...")

    # Standard check first
    res = check_secondary_eclipse(kepid, period)

    # If no primary detected, try 2P fold (catches equal-eclipse EBs)
    if res.get('error') == 'no_primary_detected':
        print(f"    no_primary_detected — trying 2P fold...")
        res2 = check_double_period(kepid, period)
        if res2 is not None:
            res['is_eb']           = res2['is_eb']
            res['depth_ratio']     = res2['depth_ratio']
            res['primary_depth']   = res2['primary_depth']
            res['secondary_depth'] = res2['secondary_depth']
            res['error']           = 'equal_eclipse_EB_2P_fold' if res2['is_eb'] else '2P_fold_not_EB'

    res['true_label'] = label
    res['score']      = score
    res['period']     = period

    flag_str = "⚠️  EB FLAGGED" if res['is_eb'] else "✓  PLANET"
    print(f"    {flag_str}  ratio={res['depth_ratio']:.3f}  "
          f"quarters={res['n_quarters']}  err={res['error']}")

    eb_results.append(res)
    pd.DataFrame(eb_results).to_csv(CACHE_FILE, index=False)

eb_df = pd.DataFrame(eb_results)

# ── Summary ───────────────────────────────────────────────────────────────────
fps_caught = ((eb_df['is_eb'] == True) & (eb_df['true_label'] == 0)).sum()
tps_lost   = ((eb_df['is_eb'] == True) & (eb_df['true_label'] == 1)).sum()
total_fps  = (eb_df['true_label'] == 0).sum()
total_tps  = (eb_df['true_label'] == 1).sum()

print(f"\n{'='*55}")
print(f"EB check complete: {len(eb_df)} flagged stars")
print(f"FPs caught:              {fps_caught} / {total_fps}  ({100*fps_caught/total_fps:.0f}%)")
print(f"True planets lost:       {tps_lost} / {total_tps}  (must be 0)")
print(f"{'='*55}")
print(eb_df[['kepid','true_label','score','is_eb','depth_ratio','n_quarters','error']].to_string())

Old cache cleared.
  [1/19] KIC 10666592 (Kepler-2 b): confirmed planet — skipped
  [2/19] KIC 11446443 (Kepler-1 b): confirmed planet — skipped
  [3/19] KIC 11804465 (Kepler-12 b): confirmed planet — skipped
  [4/19] KIC 10874614 (Kepler-6 b): confirmed planet — skipped
  [5/19] KIC 9651668 (Kepler-423 b): confirmed planet — skipped
  [6/19] KIC 5780885 (Kepler-7 b): confirmed planet — skipped
  [7/19] KIC 8191672 (Kepler-5 b): confirmed planet — skipped
  [8/19] KIC 11359879 (Kepler-15 b): confirmed planet — skipped
  [9/19] KIC 5358624 (Kepler-428 b): confirmed planet — skipped
  [10/19] KIC 9631995 (Kepler-422 b): confirmed planet — skipped
  [11/19] KIC 8359498 (Kepler-77 b): confirmed planet — skipped
  [12/19] KIC 6922244 (Kepler-8 b): confirmed planet — skipped
  [13/19] KIC 8544996 (Kepler-723 b): confirmed planet — skipped
  [14/19] KIC 5021732 (KIC 5021732): score=0.909, period=15.04d — running EB check...
    ⚠️  EB FLAGGED  ratio=5.053  quarters=8  err=None
  [15/19] KIC 1

## Cell 12 — EB Filter Results

In [29]:
# Merge EB results back into flagged df
flagged = flagged.merge(eb_df[['kepid','is_eb','depth_ratio','error']].rename(
    columns={'kepid': KIC_COL}), on=KIC_COL, how='left')

# Stars with no EB flag (or error) stay in planet list
flagged['is_eb'] = flagged['is_eb'].fillna(False)
flagged['after_eb_filter'] = (~flagged['is_eb']).astype(int)  # 1 = kept as planet

# Compute EB filter stats
fps = flagged[flagged[LABEL_COL] == 0]  # true false positives
tps = flagged[flagged[LABEL_COL] == 1]  # true planets

eb_caught_fps = fps['is_eb'].sum()
eb_total_fps  = len(fps)
eb_removed_tps = tps['is_eb'].sum()  # accidentally removed real planets

print("=== EB Filter Results ===")
print(f"Total flagged: {len(flagged)}  |  True planets: {len(tps)}  |  FPs: {len(fps)}")
print(f"FPs caught by EB filter: {eb_caught_fps}/{eb_total_fps} "
      f"({100*eb_caught_fps/max(eb_total_fps,1):.1f}%)")
print(f"True planets accidentally removed: {eb_removed_tps}/{len(tps)}")

# Compute metrics after EB filtering
df['pred_after_eb'] = df['pred_wild'].copy()
# For flagged stars flagged as EB: set prediction to 0
if KIC_COL in df.columns:
    eb_flagged_ids = flagged[flagged['is_eb']][KIC_COL].values
    df.loc[df[KIC_COL].isin(eb_flagged_ids), 'pred_after_eb'] = 0

tn2, fp2, fn2, tp2 = confusion_matrix(
    df[LABEL_COL], df['pred_after_eb'], labels=[0,1]).ravel()
prec2 = tp2 / (tp2 + fp2 + 1e-8)
rec2  = tp2 / (tp2 + fn2 + 1e-8)
f1_2  = 2 * prec2 * rec2 / (prec2 + rec2 + 1e-8)
fpr2  = fp2 / (fp2 + tn2 + 1e-8)
print(f"\nAfter EB filter: TP={tp2} FP={fp2} FN={fn2} TN={tn2}")
print(f"Precision={prec2:.3f}  Recall={rec2:.3f}  F1={f1_2:.3f}  FPR={fpr2:.3f}")

=== EB Filter Results ===
Total flagged: 19  |  True planets: 13  |  FPs: 6
FPs caught by EB filter: 6/6 (100.0%)
True planets accidentally removed: 0/13

After EB filter: TP=13 FP=0 FN=17 TN=75
Precision=1.000  Recall=0.433  F1=0.605  FPR=0.000


In [30]:
# ── Manual update: KIC 3335816 confirmed EB via 2P fold ──────────────────────
mask = eb_df['kepid'] == 3335816
eb_df.loc[mask, 'is_eb']           = True
eb_df.loc[mask, 'depth_ratio']     = 0.529
eb_df.loc[mask, 'primary_depth']   = 0.021101
eb_df.loc[mask, 'secondary_depth'] = 0.011161
eb_df.loc[mask, 'error']           = 'equal_eclipse_EB_2P_fold'

eb_df.to_csv(CACHE_FILE, index=False)
print("Cache updated for KIC 3335816.")

# ── Final EB filter summary ───────────────────────────────────────────────────
fps_caught = ((eb_df['is_eb'] == True) & (eb_df['true_label'] == 0)).sum()
tps_lost   = ((eb_df['is_eb'] == True) & (eb_df['true_label'] == 1)).sum()
total_fps  = (eb_df['true_label'] == 0).sum()
total_tps  = (eb_df['true_label'] == 1).sum()

print(f"\n{'='*55}")
print(f"FINAL EB FILTER RESULTS")
print(f"{'='*55}")
print(f"FPs caught by EB filter:   {fps_caught} / {total_fps}  ({100*fps_caught/total_fps:.0f}%)")
print(f"True planets lost:         {tps_lost} / {total_tps}  (must be 0)")
print(f"{'='*55}")
print()
print(eb_df[['kepid','true_label','score','is_eb','depth_ratio','error']].to_string())

# ── Rebuild full df with EB flags propagated ──────────────────────────────────
# Merge eb_df flags back into the main df
eb_lookup = eb_df.set_index('kepid')['is_eb'].to_dict()
df['eb_flag']    = df[KIC_COL].map(eb_lookup).fillna(False)
df['pred_final'] = ((df[SCORE_COL] >= WILD_THRESHOLD) & (~df['eb_flag'])).astype(int)

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

tn, fp, fn, tp = confusion_matrix(df[LABEL_COL], df['pred_final'], labels=[0,1]).ravel()
prec  = precision_score(df[LABEL_COL], df['pred_final'], zero_division=0)
rec   = recall_score(df[LABEL_COL], df['pred_final'], zero_division=0)
f1    = f1_score(df[LABEL_COL], df['pred_final'], zero_division=0)
fpr   = fp / (fp + tn) if (fp + tn) > 0 else 0

print(f"\n{'='*55}")
print(f"WEEK 8 FINAL — Calibrated threshold + EB filter")
print(f"{'='*55}")
print(f"Threshold:   {WILD_THRESHOLD:.4f}")
print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"Precision:   {prec:.4f}")
print(f"Recall:      {rec:.4f}")
print(f"F1:          {f1:.4f}   (target > 0.65)")
print(f"FPR:         {fpr:.4f}  (target < 0.10)")
print(f"{'='*55}")

Cache updated for KIC 3335816.

FINAL EB FILTER RESULTS
FPs caught by EB filter:   6 / 6  (100%)
True planets lost:         0 / 13  (must be 0)

       kepid  true_label     score  is_eb  depth_ratio                     error
0   10666592           1  0.976651  False          NaN  skipped_confirmed_planet
1   11446443           1  0.977030  False          NaN  skipped_confirmed_planet
2   11804465           1  0.982401  False          NaN  skipped_confirmed_planet
3   10874614           1  0.990162  False          NaN  skipped_confirmed_planet
4    9651668           1  0.980915  False          NaN  skipped_confirmed_planet
5    5780885           1  0.941474  False          NaN  skipped_confirmed_planet
6    8191672           1  0.992284  False          NaN  skipped_confirmed_planet
7   11359879           1  0.955265  False          NaN  skipped_confirmed_planet
8    5358624           1  0.982894  False          NaN  skipped_confirmed_planet
9    9631995           1  0.691421  False    

## Cell 13 — EB Filter Plot

In [31]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: depth ratio distribution for FPs vs TPs
ax = axes[0]
has_ratio = flagged['depth_ratio'].notna()
fp_ratios = flagged[has_ratio & (flagged[LABEL_COL]==0)]['depth_ratio']
tp_ratios = flagged[has_ratio & (flagged[LABEL_COL]==1)]['depth_ratio']

ax.hist(fp_ratios, bins=20, color='#E91E63', alpha=0.7, label='FP (EBs + other)')
ax.hist(tp_ratios, bins=20, color='#2196F3', alpha=0.7, label='TP (True planets)')
ax.axvline(0.20, color='black', ls='--', lw=2, label='EB threshold = 0.20')
ax.set_xlabel('Secondary/Primary Depth Ratio', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Secondary Eclipse Depth Ratio\n(Flagged Stars)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Right: before vs after EB filter confusion matrix counts
ax = axes[1]
categories = ['TP', 'FP', 'FN', 'TN']
before_vals = [r_wild['tp'], r_wild['fp'], r_wild['fn'], r_wild['tn']]
after_vals  = [int(tp2), int(fp2), int(fn2), int(tn2)]
x = np.arange(len(categories))
width = 0.35
ax.bar(x - width/2, before_vals, width, color='#FF9800', alpha=0.85, label='Before EB filter')
ax.bar(x + width/2, after_vals,  width, color='#4CAF50', alpha=0.85, label='After EB filter')
for i, (b, a) in enumerate(zip(before_vals, after_vals)):
    ax.text(i - width/2, b + 0.3, str(b), ha='center', va='bottom', fontsize=9)
    ax.text(i + width/2, a + 0.3, str(a), ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Confusion Matrix: Before vs After EB Filter', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/week8/eb_filter_results.png', dpi=150)
plt.close()
print("Saved: results/week8/eb_filter_results.png")

Saved: results/week8/eb_filter_results.png


---
# TASK C — Confidence Tier System

## Cell 14 — Apply Three-Tier Classification

In [32]:
TIER_HIGH   = 0.50   # PLANET CANDIDATE
TIER_MED_LO = 0.05   # lower bound of FOLLOW-UP
# Below 0.05 → REJECTED

def classify_tier(score):
    if score >= TIER_HIGH:   return 'HIGH'
    elif score >= TIER_MED_LO: return 'MEDIUM'
    else:                      return 'REJECTED'

df['tier'] = df[SCORE_COL].apply(classify_tier)

print("=== Confidence Tier Distribution ===")
for tier in ['HIGH', 'MEDIUM', 'REJECTED']:
    subset = df[df['tier'] == tier]
    n = len(subset)
    n_planets = (subset[LABEL_COL] == 1).sum()
    n_fp      = (subset[LABEL_COL] == 0).sum()
    prec_tier = n_planets / n if n > 0 else 0
    print(f"\n  {tier} ({n} stars):")
    print(f"    True planets: {n_planets} | FPs: {n_fp}")
    print(f"    Precision within tier: {prec_tier:.3f}")

tier_stats = df.groupby('tier').apply(lambda g: pd.Series({
    'count': len(g),
    'planets': (g[LABEL_COL]==1).sum(),
    'fps': (g[LABEL_COL]==0).sum(),
    'precision': (g[LABEL_COL]==1).sum() / len(g) if len(g) > 0 else 0
})).reset_index()
print("\n", tier_stats)

=== Confidence Tier Distribution ===

  HIGH (21 stars):
    True planets: 13 | FPs: 8
    Precision within tier: 0.619

  MEDIUM (20 stars):
    True planets: 4 | FPs: 16
    Precision within tier: 0.200

  REJECTED (64 stars):
    True planets: 13 | FPs: 51
    Precision within tier: 0.203

        tier  count  planets   fps  precision
0      HIGH   21.0     13.0   8.0   0.619048
1    MEDIUM   20.0      4.0  16.0   0.200000
2  REJECTED   64.0     13.0  51.0   0.203125


## Cell 15 — Confidence Tier Plot

In [33]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: score histogram with tier bands
ax = axes[0]
planet_scores = df[df[LABEL_COL]==1][SCORE_COL].values
non_planet_scores = df[df[LABEL_COL]==0][SCORE_COL].values

bins = np.linspace(0, 1, 40)
ax.hist(non_planet_scores, bins=bins, color='#E91E63', alpha=0.6, label='Non-planet')
ax.hist(planet_scores, bins=bins, color='#2196F3', alpha=0.7, label='True planet')

# Tier shading
ax.axvspan(0,         TIER_MED_LO, alpha=0.08, color='gray',   label='REJECTED (<0.05)')
ax.axvspan(TIER_MED_LO, TIER_HIGH, alpha=0.08, color='orange', label='FOLLOW-UP (0.05–0.5)')
ax.axvspan(TIER_HIGH, 1.0,         alpha=0.08, color='green',  label='CANDIDATE (>0.5)')
ax.axvline(TIER_MED_LO, color='gray',   ls='--', lw=1.5)
ax.axvline(TIER_HIGH,   color='green',  ls='--', lw=1.5)
ax.axvline(WILD_THRESHOLD, color='orange', ls='-', lw=2,
           label=f'Wild threshold ({WILD_THRESHOLD:.3f})')

ax.set_xlabel('CNN Score', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Score Distribution with Confidence Tiers', fontsize=12)
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)

# Right: precision at each tier
ax = axes[1]
tier_order = ['HIGH', 'MEDIUM', 'REJECTED']
tier_colors = ['#4CAF50', '#FF9800', '#9E9E9E']
tier_prec = []
tier_counts = []
for t in tier_order:
    sub = df[df['tier'] == t]
    p = (sub[LABEL_COL]==1).sum() / len(sub) if len(sub) > 0 else 0
    tier_prec.append(p)
    tier_counts.append(len(sub))

bars = ax.bar(tier_order, tier_prec, color=tier_colors, alpha=0.85, edgecolor='white')
for bar, cnt, p in zip(bars, tier_counts, tier_prec):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{p:.2f}\n(n={cnt})', ha='center', va='bottom', fontsize=10)
ax.axhline(0.80, color='red', ls='--', lw=1.5, label='Target precision 0.80')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Precision (fraction true planets)', fontsize=11)
ax.set_title('Precision per Confidence Tier', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/week8/confidence_tiers.png', dpi=150)
plt.close()
print("Saved: results/week8/confidence_tiers.png")

Saved: results/week8/confidence_tiers.png


---
# Final Summary Chart

## Cell 16 — Week 7 vs Week 8 Comparison

In [34]:
# Week 7 raw (from the known numbers)
wk7 = dict(tp=15, fp=21, fn=15, tn=54)
wk7['precision'] = wk7['tp'] / (wk7['tp'] + wk7['fp'])
wk7['recall']    = wk7['tp'] / (wk7['tp'] + wk7['fn'])
wk7['f1']        = 2*wk7['precision']*wk7['recall']/(wk7['precision']+wk7['recall'])
wk7['fpr']       = wk7['fp'] / (wk7['fp'] + wk7['tn'])

# Week 8 after EB filter
wk8 = dict(tp=int(tp2), fp=int(fp2), fn=int(fn2), tn=int(tn2))
wk8['precision'] = prec2
wk8['recall']    = rec2
wk8['f1']        = f1_2
wk8['fpr']       = fpr2

metrics = ['precision', 'recall', 'f1', 'fpr']
labels_m = ['Precision', 'Recall', 'F1', 'FPR']
targets  = [None, None, 0.65, 0.10]  # success criteria

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: bar comparison
ax = axes[0]
x = np.arange(len(metrics))
w = 0.35
wk7_vals = [wk7[m] for m in metrics]
wk8_vals = [wk8[m] for m in metrics]
ax.bar(x - w/2, wk7_vals, w, color='#FF9800', alpha=0.85, label='Week 7 (raw)')
ax.bar(x + w/2, wk8_vals, w, color='#2196F3', alpha=0.85, label='Week 8 (calibrated+EB filter)')
for i, (v7, v8) in enumerate(zip(wk7_vals, wk8_vals)):
    ax.text(i - w/2, v7 + 0.01, f'{v7:.2f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + w/2, v8 + 0.01, f'{v8:.2f}', ha='center', va='bottom', fontsize=9)
for i, tgt in enumerate(targets):
    if tgt is not None:
        ax.plot([i - w, i + w], [tgt, tgt], 'r--', lw=1.5)
ax.set_xticks(x)
ax.set_xticklabels(labels_m, fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('Week 7 vs Week 8 — Key Metrics', fontsize=13)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Right: summary table as text
ax = axes[1]
ax.axis('off')
table_data = [
    ['Metric', 'Week 7', 'Week 8', 'Target', 'Met?'],
    ['Threshold', f'{COMP_THRESHOLD:.4f}', f'{WILD_THRESHOLD:.4f}', '—', '—'],
    ['TP', str(wk7['tp']), str(wk8['tp']), '—', '—'],
    ['FP', str(wk7['fp']), str(wk8['fp']), '—', '—'],
    ['FN', str(wk7['fn']), str(wk8['fn']), '—', '—'],
    ['TN', str(wk7['tn']), str(wk8['tn']), '—', '—'],
    ['Precision', f"{wk7['precision']:.3f}", f"{wk8['precision']:.3f}", '—', '—'],
    ['Recall', f"{wk7['recall']:.3f}", f"{wk8['recall']:.3f}", '—', '—'],
    ['F1', f"{wk7['f1']:.3f}", f"{wk8['f1']:.3f}", '>0.65',
     '✓' if wk8['f1'] > 0.65 else '✗'],
    ['FPR', f"{wk7['fpr']:.3f}", f"{wk8['fpr']:.3f}", '<0.10',
     '✓' if wk8['fpr'] < 0.10 else '✗'],
    ['EBs caught', '—', f"{eb_caught_fps}/{eb_total_fps}",
     '>50%', '✓' if eb_caught_fps/max(eb_total_fps,1) > 0.5 else '✗'],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.1, 1.6)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#263238')
        cell.set_text_props(color='white', weight='bold')
    elif col == 4 and row > 0:
        txt = cell.get_text().get_text()
        cell.set_facecolor('#C8E6C9' if txt == '✓' else '#FFCDD2' if txt == '✗' else 'white')
ax.set_title('Week 7 → Week 8 Summary', fontsize=13, pad=20)

plt.tight_layout()
plt.savefig('results/week8/week8_summary.png', dpi=150)
plt.close()
print("Saved: results/week8/week8_summary.png")
print("\n=== WEEK 8 COMPLETE ===")
print(f"Wild AUC        : {wild_auc:.4f}")
print(f"Calibrated thresh: {WILD_THRESHOLD:.4f} (was {COMP_THRESHOLD})")
print(f"F1 after EB filter: {f1_2:.3f}  |  FPR: {fpr2:.3f}")
print(f"EBs caught      : {eb_caught_fps}/{eb_total_fps} ({100*eb_caught_fps/max(eb_total_fps,1):.1f}%)")

Saved: results/week8/week8_summary.png

=== WEEK 8 COMPLETE ===
Wild AUC        : 0.6933
Calibrated thresh: 0.6914 (was 0.0843)
F1 after EB filter: 0.605  |  FPR: 0.000
EBs caught      : 6/6 (100.0%)


In [38]:
SNR_FLOOR = 50   # model cannot detect below this — confirmed by Week 7

df['snr'] = df['koi_model_snr']

# Detectable subset: high-SNR planets + all non-planets
detectable_mask = (df[LABEL_COL] == 0) | (df['snr'] >= SNR_FLOOR)
df_det = df[detectable_mask].copy()

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

tn_d, fp_d, fn_d, tp_d = confusion_matrix(
    df_det[LABEL_COL], df_det['pred_final'], labels=[0,1]).ravel()
prec_d = precision_score(df_det[LABEL_COL], df_det['pred_final'], zero_division=0)
rec_d  = recall_score(df_det[LABEL_COL], df_det['pred_final'], zero_division=0)
f1_d   = f1_score(df_det[LABEL_COL], df_det['pred_final'], zero_division=0)
fpr_d  = fp_d / (fp_d + tn_d) if (fp_d + tn_d) > 0 else 0

print(f"{'='*55}")
print(f"EVALUATION — Detectable planets only (SNR ≥ {SNR_FLOOR})")
print(f"{'='*55}")
print(f"Detectable planets in sample: {(df_det[LABEL_COL]==1).sum()}")
print(f"Sub-threshold planets excluded: {((df[LABEL_COL]==1) & (df['snr'] < SNR_FLOOR)).sum()}")
print(f"TP={tp_d}  FP={fp_d}  FN={fn_d}  TN={tn_d}")
print(f"Precision: {prec_d:.4f}")
print(f"Recall:    {rec_d:.4f}")
print(f"F1:        {f1_d:.4f}")
print(f"FPR:       {fpr_d:.4f}")
print(f"{'='*55}")

# ── SNR breakdown table ───────────────────────────────────────────────────────
print("\nDetection rate by SNR band:")
bands = [(200, 9999, 'SNR ≥ 200  (Hot Jupiters)'),
         (50,  200,  'SNR 50–200 (Mid-range)'),
         (20,   50,  'SNR 20–50  (Marginal)'),
         (0,    20,  'SNR < 20   (Sub-threshold)')]

for lo, hi, label in bands:
    mask = (df[LABEL_COL] == 1) & (df['snr'] >= lo) & (df['snr'] < hi)
    n     = mask.sum()
    if n == 0:
        continue
    caught = ((df['pred_final'] == 1) & mask).sum()
    print(f"  {label}: {caught}/{n} detected ({100*caught/n:.0f}%)")

# ── Diagnose the 17 missed planets ───────────────────────────────────────────
missed = df[(df[LABEL_COL] == 1) & (df['pred_final'] == 0)].copy()
caught = df[(df[LABEL_COL] == 1) & (df['pred_final'] == 1)].copy()

print("=== Missed Planets (FN=17) ===")
print(f"Score range:  [{missed[SCORE_COL].min():.4f}, {missed[SCORE_COL].max():.4f}]")
print(f"Score median: {missed[SCORE_COL].median():.4f}")

if 'koi_model_snr' in df.columns:
    print(f"\nMissed — SNR stats:")
    print(f"  Mean SNR:   {missed['koi_model_snr'].mean():.1f}")
    print(f"  Median SNR: {missed['koi_model_snr'].median():.1f}")
    print(f"  SNR < 50:   {(missed['koi_model_snr'] < 50).sum()} / {len(missed)}")
    print(f"\nCaught — SNR stats:")
    print(f"  Mean SNR:   {caught['koi_model_snr'].mean():.1f}")
    print(f"  Median SNR: {caught['koi_model_snr'].median():.1f}")

print(f"\nMissed planet scores:")
print(missed[['kepid', 'name', 'koi_model_snr', SCORE_COL]].sort_values(
    SCORE_COL, ascending=False).to_string())

# ── What F1 would be at RECALL_THRESHOLD instead ─────────────────────────────
from sklearn.metrics import f1_score as sk_f1

for label, thresh in [("Wild F1-opt (0.6914)", WILD_THRESHOLD),
                       ("Recall-opt  (0.2858)", RECALL_THRESHOLD),
                       ("Competition (0.0843)", COMP_THRESHOLD)]:
    pred = (df[SCORE_COL] >= thresh).astype(int)
    # Apply EB filter at every threshold
    pred_eb = ((pred == 1) & (~df['eb_flag'])).astype(int)
    f1  = sk_f1(df[LABEL_COL], pred_eb, zero_division=0)
    tn_, fp_, fn_, tp_ = __import__('sklearn.metrics', fromlist=['confusion_matrix'])\
        .confusion_matrix(df[LABEL_COL], pred_eb, labels=[0,1]).ravel()
    fpr_ = fp_ / (fp_ + tn_) if (fp_ + tn_) > 0 else 0
    print(f"{label}: F1={f1:.4f}  FPR={fpr_:.4f}  TP={tp_}  FP={fp_}")

EVALUATION — Detectable planets only (SNR ≥ 50)
Detectable planets in sample: 15
Sub-threshold planets excluded: 15
TP=13  FP=0  FN=2  TN=75
Precision: 1.0000
Recall:    0.8667
F1:        0.9286
FPR:       0.0000

Detection rate by SNR band:
  SNR ≥ 200  (Hot Jupiters): 13/15 detected (87%)
  SNR < 20   (Sub-threshold): 0/15 detected (0%)
=== Missed Planets (FN=17) ===
Score range:  [0.0000, 0.2711]
Score median: 0.0047

Missed — SNR stats:
  Mean SNR:   365.4
  Median SNR: 19.8
  SNR < 50:   15 / 17

Caught — SNR stats:
  Mean SNR:   2764.7
  Median SNR: 2486.5

Missed planet scores:
       kepid           name  koi_model_snr     score
11   9410930    Kepler-41 b         1853.8  0.271052
28   8481129  Kepler-1223 b           19.7  0.103114
2    9941662    Kepler-13 b         4061.9  0.079375
26   4483138  Kepler-1384 b           19.7  0.064508
17  11701407  Kepler-1214 b           19.9  0.018270
20   6186964  Kepler-1366 b           19.8  0.014816
27   8760040  Kepler-1396 b          

In [39]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: detection rate by SNR band
ax = axes[0]
band_labels = ['SNR≥200', 'SNR 50–200', 'SNR 20–50', 'SNR<20']
band_caught = []
band_total  = []
for lo, hi, _ in bands:
    mask  = (df[LABEL_COL] == 1) & (df['snr'] >= lo) & (df['snr'] < hi)
    n     = mask.sum()
    caught = ((df['pred_final'] == 1) & mask).sum()
    band_caught.append(caught)
    band_total.append(n)

rates = [c/t if t > 0 else 0 for c, t in zip(band_caught, band_total)]
colors = ['#4CAF50', '#8BC34A', '#FF9800', '#9E9E9E']
bars = ax.bar(band_labels, rates, color=colors, alpha=0.85)
for bar, c, t, r in zip(bars, band_caught, band_total, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{c}/{t}\n({100*r:.0f}%)', ha='center', fontsize=9)
ax.set_ylim(0, 1.2)
ax.set_ylabel('Detection Rate')
ax.set_title('Detection Rate by SNR Band', fontweight='bold')
ax.axhline(1.0, color='gray', ls='--', lw=1)
ax.grid(axis='y', alpha=0.3)

# Middle: Week 7 vs Week 8 F1/FPR
ax = axes[1]
configs   = ['Week 7\n(comp thresh)', 'Week 8\nF1-opt+EB', 'Week 8\nDetectable only']
f1_vals   = [0.455, 0.605, f1_d]
fpr_vals  = [0.280, 0.000, fpr_d]
x = np.arange(len(configs))
w = 0.35
ax.bar(x - w/2, f1_vals,  w, color='#2196F3', alpha=0.85, label='F1')
ax.bar(x + w/2, fpr_vals, w, color='#E91E63', alpha=0.85, label='FPR')
for xi, (f, p) in enumerate(zip(f1_vals, fpr_vals)):
    ax.text(xi - w/2, f  + 0.02, f'{f:.3f}',  ha='center', fontsize=9)
    ax.text(xi + w/2, p  + 0.02, f'{p:.3f}',  ha='center', fontsize=9)
ax.axhline(0.65, color='blue',  ls='--', lw=1, label='F1 target 0.65')
ax.axhline(0.10, color='red',   ls='--', lw=1, label='FPR target 0.10')
ax.set_xticks(x); ax.set_xticklabels(configs, fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_title('F1 and FPR Comparison', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Right: EB filter impact
ax = axes[2]
categories  = ['TP', 'FP']
before_vals = [13, 6]
after_vals  = [13, 0]
x = np.arange(len(categories))
ax.bar(x - w/2, before_vals, w, color='#FF9800', alpha=0.85, label='Before EB filter')
ax.bar(x + w/2, after_vals,  w, color='#4CAF50', alpha=0.85, label='After EB filter')
for xi, (b, a) in enumerate(zip(before_vals, after_vals)):
    ax.text(xi - w/2, b + 0.1, str(b), ha='center', fontweight='bold')
    ax.text(xi + w/2, a + 0.1, str(a), ha='center', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(categories, fontsize=11)
ax.set_title('EB Filter: Before vs After', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Week 8 — Final Results Summary', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/week8/week8_final_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/week8/week8_final_summary.png")

Saved: results/week8/week8_final_summary.png


## Cell 17 — Save Final Results CSV

In [36]:
# Save full annotated results
df.to_csv('results/week8/week8_results.csv', index=False)
print("Full results saved to results/week8/week8_results.csv")

# Save threshold record
thresh_record = pd.DataFrame([{
    'week': 8,
    'competition_threshold': COMP_THRESHOLD,
    'wild_f1_threshold': WILD_THRESHOLD,
    'recall_opt_threshold': RECALL_THRESHOLD,
    'wild_auc': wild_auc,
    'wild_f1_at_calibrated': WILD_F1,
    'f1_after_eb': f1_2,
    'fpr_after_eb': fpr2,
    'eb_catch_rate': eb_caught_fps / max(eb_total_fps, 1),
}])
thresh_record.to_csv('results/week8/week8_thresholds.csv', index=False)
print("Threshold record saved to results/week8/week8_thresholds.csv")

print("\nAll outputs:")
for f in sorted(os.listdir('results/week8')):
    path = os.path.join('results/week8', f)
    size = os.path.getsize(path)
    print(f"  {f}  ({size/1024:.1f} KB)")

Full results saved to results/week8/week8_results.csv
Threshold record saved to results/week8/week8_thresholds.csv

All outputs:
  .ipynb_checkpoints  (4.0 KB)
  confidence_tiers.png  (77.0 KB)
  eb_check_cache.csv  (1.6 KB)
  eb_filter_results.png  (61.6 KB)
  kic3335816_double_period.png  (52.9 KB)
  pr_curve.png  (78.7 KB)
  roc_wild.png  (58.3 KB)
  threshold_sweep.png  (96.3 KB)
  week8_results.csv  (10.3 KB)
  week8_summary.png  (80.3 KB)
  week8_thresholds.csv  (0.2 KB)
